#Bronze layer

## Read first csv file

Read files in bronze layer, keep their headers(Header) and datatypes(inferShema)

In [0]:
df = (spark.read.option('header', True).option('inferSchema', True).csv("/Volumes/db_project/bronze/source_files/source_crm/cust_info.csv"))
df.display()

## Write dataframe into table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("db_project.bronze.crm_cust_info")
df.printSchema()

## Write rest of files as delta tables using loop

Create tables using for loops for the rest of the files\
(added .option("overwriteSchema", "true") because of "inferSchema" misspelling and creating incorrect tables at first)\
(All columns were String type instead of correct data types)

In [0]:
files_crm = [
    "prd_info",
    "sales_details"
]
files_erp = [
    "CUST_AZ12",
    "LOC_A101",
    "PX_CAT_G1V2"
]

for file in files_crm:
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"/Volumes/db_project/bronze/source_files/source_crm/{file}.csv")
    
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(f"db_project.bronze.crm_{file}")
    df.printSchema()

for file in files_erp:
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"/Volumes/db_project/bronze/source_files/source_erp/{file}.csv")
    
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(f"db_project.bronze.erp_{file}")
    df.printSchema()

## See the first 5 rows from each table

In [0]:
files = [
    "crm_cust_info",
    "crm_prd_info",
    "crm_sales_details",
    "erp_CUST_AZ12",
    "erp_LOC_A101",
    "erp_PX_CAT_G1V2"
]
for file in files:
    result = spark.sql(
            f"""SELECT * 
                FROM db_project.bronze.{file}
                LIMIT 5""")
    result.display()